# M2.7 · k-means

En basit kümeleme. Dört noktada iki Lloyd iterasyonunu elle yapıyor, saf Python'da atama/güncelleme döngüsünü yazıyor, NumPy'da one-hot matris çarpımıyla merkezleri döngüsüz güncelliyor ve k-means++ başlatmasını kuruyor, PyTorch'ta Lloyd adımının aslında inertia'nın gradyanını sıfırlayan bir Newton adımı olduğunu autograd ile gösteriyor ve en sonda R rastgele başlatmayı tek tensörde koşturuyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m2-7-kmeans.html](https://m-gonul.github.io/pytorch-lab/m2-7-kmeans.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m2_7_kmeans.py
CHECKS = {
 "b": "import random\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy/torch kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\", \"sklearn\"}))\n    P = [(0, 0), (1, 0), (4, 4), (5, 4)]\n\n    def hand():\n        C, lab, J = need(ns, \"kmeans\")(P, [(0.0, 0.0), (1.0, 0.0)], 2)\n        close([list(c) for c in C], [[0.5, 0.0], [4.5, 4.0]]); ensure(list(lab) == [0, 0, 1, 1], f\"etiketler {lab}\"); close(J, 1.0)\n    R.test(\"elle örnek: merkezler, etiketler, inertia 1.0\", hand)\n\n    def tie():\n        ensure(list(need(ns, \"assign\")([(1, 0)], [(0, 0), (2, 0)])) == [0], \"eşitlikte küçük indeks\")\n        new = need(ns, \"update\")([(0, 0), (1, 1)], [0, 0], [(5, 5), (9, 9)])\n        close([list(c) for c in new], [[0.5, 0.5], [9, 9]], what=\"boş küme eski merkezini korur\")\n    R.test(\"eşitlik ve boş küme kuralları\", tie)\n\n    def mono():\n        rng = random.Random(0)\n        pts = [(rng.gauss(cx, 0.5), rng.gauss(cy, 0.5)) for cx, cy in [(0, 0), (5, 5), (0, 5)] for _ in range(20)]\n        C0 = [pts[0], pts[1], pts[2]]\n        Js = [need(ns, \"kmeans\")(pts, C0, it)[2] for it in range(6)]\n        ensure(all(a >= b - 1e-9 for a, b in zip(Js, Js[1:])), f\"inertia arttı: {Js}\")\n        return \"inertia (0–5 iterasyon): \" + \", \".join(f\"{v:.1f}\" for v in Js)\n    R.test(\"inertia iterasyonlarla artmıyor\", mono)\n",
 "c": "import numpy as np\nfrom labcheck import close, max_loops, no_imports, need, ensure\n\n\ndef ref(X, C, iters):\n    C = C.copy()\n    for _ in range(iters):\n        lab = ((X[:, None] - C[None]) ** 2).sum(-1).argmin(1)\n        for k in range(len(C)):\n            if (lab == k).any():\n                C[k] = X[lab == k].mean(0)\n    D = ((X[:, None] - C[None]) ** 2).sum(-1)\n    lab = D.argmin(1)\n    return C, lab, D[np.arange(len(X)), lab].sum()\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"kmeans_np: yalnız iterasyon döngüsü\", lambda: max_loops(src, 1, func=\"kmeans_np\"))\n    R.test(\"torch/sklearn kullanılmıyor\", lambda: no_imports(src, {\"torch\", \"sklearn\"}))\n    rng = np.random.default_rng(0)\n    X = np.concatenate([rng.normal(c, 0.6, (40, 2)) for c in ([0, 0], [5, 5], [0, 5], [5, 0])])\n\n    def km():\n        C0 = X[[0, 1, 2, 3]].copy()\n        C, lab, J = need(ns, \"kmeans_np\")(X, C0, 8)\n        Cr, labr, Jr = ref(X, C0, 8)\n        close(C, Cr); ensure((np.asarray(lab) == labr).all(), \"etiketler farklı\"); close(J, Jr)\n        C, lab, J = need(ns, \"kmeans_np\")(np.array([[0., 0], [1, 0], [4, 4], [5, 4]]), np.array([[0., 0], [1, 0]]), 2)\n        close(J, 1.0, what=\"elle örnek\")\n    R.test(\"kmeans_np referansla aynı (4 kümenin 1. kümeden başlatılması)\", km)\n\n    def empty():\n        C, lab, J = need(ns, \"kmeans_np\")(np.array([[0., 0], [1, 1]]), np.array([[0.5, 0.5], [100., 100]]), 3)\n        close(C, [[0.5, 0.5], [100, 100]], what=\"boş küme eski merkezini korur\")\n    R.test(\"boş küme\", empty)\n\n    def pp():\n        C = need(ns, \"kmeanspp\")(X, 4, np.random.default_rng(3))\n        r = np.random.default_rng(3)\n        want = [X[r.integers(len(X))]]\n        for _ in range(3):\n            D2 = ((X[:, None] - np.array(want)[None]) ** 2).sum(-1).min(1)\n            want.append(X[r.choice(len(X), p=D2 / D2.sum())])\n        close(C, np.array(want))\n        Jpp = np.mean([need(ns, \"kmeans_np\")(X, need(ns, \"kmeanspp\")(X, 4, np.random.default_rng(s)), 10)[2] for s in range(10)])\n        Jrnd = np.mean([need(ns, \"kmeans_np\")(X, X[np.random.default_rng(s).choice(len(X), 4, replace=False)], 10)[2] for s in range(10)])\n        return f\"10 tohum ortalaması inertia: k-means++ {Jpp:.1f}, rastgele {Jrnd:.1f}\"\n    R.test(\"kmeanspp: belirtilen rng dizisi\", pp)\n",
 "d": "import torch\nfrom labcheck import close, need, ensure, uses_any\n\n\ndef mean_update(X, C):\n    lab = ((X[:, None] - C[None]) ** 2).sum(-1).argmin(1)\n    C = C.clone()\n    for k in range(len(C)):\n        if (lab == k).any():\n            C[k] = X[lab == k].mean(0)\n    return C\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    g = torch.Generator().manual_seed(0)\n    X = torch.cat([torch.randn(30, 3, generator=g, dtype=torch.float64) + c for c in (0.0, 4.0, -4.0)])\n\n    def km():\n        C0 = X[[0, 1, 2]].clone()\n        C, lab, J = need(ns, \"kmeans_torch\")(X, C0, 10)\n        Cr = C0.clone()\n        for _ in range(10):\n            Cr = mean_update(X, Cr)\n        close(C, Cr)\n        close(J, ((X - Cr[lab]) ** 2).sum(), what=\"inertia\")\n    R.test(\"kmeans_torch referans Lloyd ile aynı\", km)\n    R.test(\"autograd kullanılıyor\", lambda: ensure(uses_any(src, [\"grad\", \"backward\"]), \"lloyd_step_via_grad autograd kullanmalı\"))\n\n    def ng():\n        f = need(ns, \"lloyd_step_via_grad\")\n        close(f(torch.tensor([[0.0, 0], [1, 0], [4, 4], [5, 4]], dtype=torch.float64), torch.tensor([[0.0, 0], [1, 0]], dtype=torch.float64)), [[0, 0], [10 / 3, 8 / 3]], what=\"elle örnek\")\n        C = torch.randn(4, 3, generator=g, dtype=torch.float64) * 3\n        C[3] = 100.0\n        close(f(X, C), mean_update(X, C), what=\"rastgele merkezler + boş küme\")\n        return \"Newton adımı c − ∇J/(2nₖ) = küme ortalaması\"\n    R.test(\"lloyd_step_via_grad = ortalama güncellemesi\", ng)\n",
 "e": "import torch\nfrom labcheck import close, max_loops, need, ensure, bench, fmt_time\n\n\ndef one(X, C, iters):\n    C = C.clone(); k = C.shape[0]\n    for _ in range(iters):\n        lab = torch.cdist(X, C).argmin(1)\n        s = torch.zeros_like(C).index_add_(0, lab, X)\n        c = torch.bincount(lab, minlength=k).to(X.dtype)\n        C = torch.where(c[:, None] > 0, s / c.clamp_min(1)[:, None], C)\n    return C, (torch.cdist(X, C) ** 2).min(1).values.sum()\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"yalnız iterasyon döngüsü\", lambda: max_loops(src, 1, func=\"kmeans_restarts\", why=\"başlatmalar üzerinde döngü olmamalı\"))\n    g = torch.Generator().manual_seed(0)\n    X = torch.cat([torch.randn(40, 2, generator=g, dtype=torch.float64) + torch.tensor(c, dtype=torch.float64) for c in ([0.0, 0], [6, 0], [0, 6], [6, 6])])\n    C0s = X[torch.randint(0, 160, (10, 4), generator=g)]\n\n    def eq():\n        best, J = need(ns, \"kmeans_restarts\")(X, C0s, 15)\n        ensure(tuple(J.shape) == (10,), f\"inertias shape {tuple(J.shape)}\")\n        ref = [one(X, C0s[r], 15) for r in range(10)]\n        close(J, torch.stack([j for _, j in ref]), rtol=1e-9)\n        close(best, ref[int(torch.stack([j for _, j in ref]).argmin())][0])\n        return f\"inertia aralığı {float(J.min()):.1f} – {float(J.max()):.1f}: başlatma önemli\"\n    R.test(\"10 başlatma: tek tek koşturmayla aynı\", eq)\n    if \"kmeans_restarts\" not in ns:\n        return\n    Xb = torch.randn(2000, 8, generator=g)\n    C0b = Xb[torch.randint(0, 2000, (64, 10), generator=g)]\n    t_loop = bench(lambda: [one(Xb, C0b[r], 10) for r in range(64)], repeat=3, number=1)\n    t_vec = bench(lambda: ns[\"kmeans_restarts\"](Xb, C0b, 10), repeat=3, number=1)\n    R.table(\"64 başlatma × 10 iterasyon (n = 2000, k = 10)\", [\"yöntem\", \"süre\", \"döngüye göre\"],\n            [[\"for r: kmeans\", fmt_time(t_loop), \"1×\"], [\"senin kmeans_restarts\", fmt_time(t_vec), f\"{t_loop / t_vec:,.1f}× hızlı\"]])\n"
}

In [ ]:
import importlib, labcheck, checks_m2_7_kmeans as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

$k$ merkez $c_1,\dots,c_k$ ve her noktanın atandığı küme $z_i$ için amaç inertia:

$$
J(C,z)=\sum_{i=1}^n\lVert x_i-c_{z_i}\rVert^2 .
$$

**Lloyd algoritması** iki adımı tekrarlar:

1. **Ata:** $z_i=\arg\min_k\lVert x_i-c_k\rVert^2$ (eşitlikte küçük $k$).
2. **Güncelle:** $c_k=\frac1{n_k}\sum_{i:z_i=k}x_i$. Küme boşsa ($n_k=0$) merkezi değiştirme.

Güncelleme adımının matris hâli: atama matrisi $A\in\{0,1\}^{n\times k}$ (one-hot) ile $C=\text{diag}(A^\top\mathbf 1)^{-1}A^\top X$. Atama adımı ise M2.1'deki mesafe matrisinin satır argmin'i.

Sonuç başlangıca bağlıdır. **k-means++**: ilk merkez rastgele; sonraki her merkez, noktaları en yakın mevcut merkeze kare uzaklığı $D(x)^2$ ile orantılı olasılıkla seçer.

## 1 · Elle

Dört noktayı kareli kâğıda çiz. Başlangıç merkezleri kötü seçilmiş: ikisi de sol alttaki gruba yakın. İlk iterasyonda $c_2$ üç noktayı alıp sağa kayıyor, ikinci iterasyonda $(1,0)$ $c_1$'e geri dönüyor ve algoritma doğru iki kümeyi buluyor.

Bu sefer şanslıydık. $c_1=(0,0)$, $c_2=(4,4)$ ve beşinci bir nokta $(5,4)$ yerine $(10,10)$ olsaydı bir merkez aykırı değere kilitlenebilirdi; k-means++ ve çoklu başlatma bu yüzden var.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**Noktalar $(0,0),(1,0),(4,4),(5,4)$; başlangıç merkezleri $c_1=(0,0)$, $c_2=(1,0)$. 1. iterasyon**

- `a3`: $(4,4)$'ün kümesi (1 ya da 2)
- `c2x`: güncellenmiş $c_2$'nin x'i
- `c2y`: güncellenmiş $c_2$'nin y'si

**2. iterasyon**

- `a2`: $(1,0)$'ın yeni kümesi
- `c1x`: güncellenmiş $c_1$'in x'i
- `inertia`: inertia $\sum_i\lVert x_i-c_{z_i}\rVert^2$

In [ ]:
cevaplar = dict(
    a3=None,
    c2x=None,
    c2y=None,
    a2=None,
    c1x=None,
    inertia=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"a3":[2,0],"c2x":[3.333333,0.002],"c2y":[2.666667,0.002],"a2":[1,0],"c1x":[0.5,0.001],"inertia":[1,0.001]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python; noktalar ve merkezler tuple/liste.

- `assign(points, centers)` → etiket listesi (0-tabanlı, eşitlikte küçük indeks).
- `update(points, labels, centers)` → yeni merkezler; boş küme eski merkezini korur.
- `kmeans(points, centers, iters)` → `(centers, labels, inertia)`; `iters` kez ata+güncelle, sonra son merkezlere göre son atama ve inertia.

In [ ]:
%%writefile stage_b.py
def assign(points, centers):
    ...


def update(points, labels, centers):
    ...


def kmeans(points, centers, iters):
    ...


P = [(0, 0), (1, 0), (4, 4), (5, 4)]
print(kmeans(P, [(0.0, 0.0), (1.0, 0.0)], 2))   # ([(0.5, 0.0), (4.5, 4.0)], [0, 0, 1, 1], 1.0)

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

- `kmeans_np(X, C0, iters)` → `(C, labels, inertia)`. Atama: mesafe açılımı + `argmin(1)`. Güncelleme: `A = np.eye(k)[labels]`, `sums = A.T @ X`, `counts = A.sum(0)`; boş kümede eski merkez (`np.where`). Yalnız iterasyon döngüsü.
- `kmeanspp(X, k, rng)` → `(k, d)` başlangıç merkezleri. Rastgele çağrıların sırası tam olarak: önce `rng.integers(n)`, sonra her yeni merkez için `rng.choice(n, p=D2 / D2.sum())`; $D^2$ her noktanın mevcut merkezlere en küçük kare uzaklığı. Merkezler üzerinde tek döngü serbest.

In [ ]:
%%writefile stage_c.py
import numpy as np


def kmeans_np(X, C0, iters):
    """→ (C, labels, inertia). Yalnız iterasyon döngüsü."""
    ...


def kmeanspp(X, k, rng):
    """→ (k, d). rng çağrı sırası: integers(n), sonra her merkez için choice(n, p=…)."""
    ...


rng = np.random.default_rng(0)
X = np.concatenate([rng.normal(c, 0.5, (50, 2)) for c in ([0, 0], [5, 5], [0, 5])])
C0 = kmeanspp(X, 3, np.random.default_rng(1))
print(kmeans_np(X, C0, 10)[2])

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

Lloyd güncellemesi neden ortalama? Atamalar sabitken $J$, $c_k$'ye göre bir kuadratik:

$$
\frac{\partial J}{\partial c_k}=2\sum_{i:z_i=k}(c_k-x_i)=2\big(n_kc_k-S_k\big)=0\ \Rightarrow\ c_k=\frac{S_k}{n_k}.
$$

Yani güncelleme, gradyanı sıfırlayan tam bir **Newton adımı**: $c_k\leftarrow c_k-\frac{1}{2n_k}\frac{\partial J}{\partial c_k}$.

- `kmeans_torch(X, C0, iters)` → `(C, labels, inertia)`; atama `torch.cdist(X, C).argmin(1)`, toplamlar `torch.zeros_like(C).index_add_(0, labels, X)`, sayımlar `torch.bincount(labels, minlength=k)`.
- `lloyd_step_via_grad(X, C)` → yeni merkezler; atamaları hesapla, $J$'yi **autograd** ile türet, Newton adımını uygula (boş kümede değişiklik yok). Kontrol, sonucun ortalama güncellemesiyle aynı olduğunu doğrular.

In [ ]:
%%writefile stage_d.py
import torch


def kmeans_torch(X, C0, iters):
    ...


def lloyd_step_via_grad(X, C):
    """Atamaları sabitle, inertia'nın gradyanından Newton adımı."""
    ...


X = torch.tensor([[0.0, 0], [1, 0], [4, 4], [5, 4]])
print(kmeans_torch(X, X[:2].clone(), 2))
print(lloyd_step_via_grad(X, torch.tensor([[0.0, 0], [1, 0]])))   # [[0, 0], [3.33, 2.67]]

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

Yerel minimum riskine karşı standart çözüm: $R$ farklı başlatmayla koş, en düşük inertia'yı seç. $R$'yi bir batch boyutu yap:

- merkezler `(R, k, d)`, mesafeler `(R, n, k)`, atamalar `(R, n)`,
- one-hot `(R, n, k)`, toplamlar `torch.einsum("rnk,nd->rkd", A, X)`, sayımlar `A.sum(1)` `(R, k)`.

- `kmeans_restarts(X, C0s, iters)` → `(best_C: (k, d), inertias: (R,))`. Yalnız iterasyon döngüsü; başlatmalar üzerinde döngü yok. Boş küme eski merkezini korur.

Kontrol, her başlatmayı `kmeans_torch` ile tek tek koşturup karşılaştırır.

In [ ]:
%%writefile stage_e.py
import torch
import torch.nn.functional as F


def kmeans_restarts(X, C0s, iters):
    """C0s: (R, k, d) → (best_C: (k, d), inertias: (R,)). Yalnız iterasyon döngüsü."""
    ...


g = torch.Generator().manual_seed(0)
X = torch.cat([torch.randn(50, 2, generator=g) + torch.tensor(c) for c in ([0.0, 0], [6, 0], [0, 6], [6, 6])])
C0s = X[torch.randint(0, 200, (32, 4), generator=g)]
best, J = kmeans_restarts(X, C0s, 20)
print(J.min(), J.max())

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Makale ödevi: Arthur & Vassilvitskii (2007): k-means++

[Arthur & Vassilvitskii — k-means++: The Advantages of Careful Seeding (SODA 2007)](https://theory.stanford.edu/~sergei/papers/kMeansPP-soda.pdf)

k-means++ tohumlamasını (D² ağırlıklı örnekleme) yaz ve rastgele tohumlamayla karşılaştır: makalenin Tablo 1–4'teki gibi sentetik (iyi ayrılmış Gauss'lar) ve gerçek veride ortalama/en iyi potansiyel φ ve süre. O(log k) yaklaşım garantisini sentetik veride, bilinen optimuma göre oranla deneysel olarak kontrol et; tohumlamanın tek başına (Lloyd yinelemesiz) ne kadar iyi olduğunu ölç.


**Adımlar**

1. 20 tekrar üzerinden ortalama ve en iyi φ.

**Kabul ölçütleri**

- k-means++ hem φ'de hem yinelenme sayısında rastgeleden iyi.
- Optimuma oran log k'nin çok altında.

## Kaynaklar

- [ISLP — §12.4.1 K-Means Clustering](https://www.statlearning.com/) — doğrulanmadı
- [Arthur & Vassilvitskii — k-means++: The Advantages of Careful Seeding](https://theory.stanford.edu/~sergei/papers/kMeansPP-soda.pdf) — ©
- [Stanford CS229 — Notes: The k-means clustering algorithm](https://cs229.stanford.edu/main_notes.pdf) — doğrulanmadı
- [PyTorch — Tensor.index_add_, torch.cdist](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.index_add_.html) — BSD-3